# Federated Averaging (FedAvg) on CIFAR-10 - No Rotations, Dirichlet Label Distribution

This notebook implements standard Federated Averaging (FedAvg) on CIFAR-10 data **without any rotation transformations**. All clients receive the same standard preprocessing.

**Heterogeneity:**
- **No rotation-based feature heterogeneity** (all clients use same preprocessing)
- **Label heterogeneity via Dirichlet distribution** (α parameter controls non-IID level)


**Purpose:**Baseline to compare with rotation-based methods - isolates label heterogeneity impact.

In [ ]:
# Setup for Google Colab
try:
    import google.colab
    IN_COLAB = True
    print("Running in Google Colab")
    
    from google.colab import drive
    drive.mount('/content/drive')
    
    import os
    os.chdir('/content/drive/MyDrive/EnsembleFederatedLearning')
    
    !pip install -q torch torchvision scikit-learn matplotlib seaborn
    
except ImportError:
    IN_COLAB = False
    print("Running locally")

## Import Libraries

In [ ]:
import sys
import json
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from torch.utils.data import DataLoader, Subset, Dataset
import copy
import random
import time

sys.path.append('..')
from training.utils import get_model, set_seed

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Load Configuration

In [ ]:
# Load configuration from JSON
with open('config.json', 'r') as f:
    CONFIG = json.load(f)

# Add Dirichlet alpha parameter
CONFIG['dirichlet_alpha'] = 0.5  # ← TUNE THIS: 0.1 (high non-IID) to 10.0 (low non-IID)

# Set random seeds
SEED = CONFIG['seed']
set_seed(SEED)


print("Configuration:")   
for k, v in CONFIG.items():
    print(f"  {k}: {v}")

## Load and Prepare CIFAR-10 Dataset

**No rotation transformations applied** - all clients use standard preprocessing.

In [ ]:
# Standard transform (no rotation)
standard_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

# Load CIFAR-10
print("Loading CIFAR-10 dataset...")
full_train_dataset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=standard_transform)
test_dataset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=standard_transform)

print(f"Full train dataset size: {len(full_train_dataset)}")
print(f"Test dataset size: {len(test_dataset)}")

# Split train into train (80%) and validation (20%)

train_size = int(0.8 * len(full_train_dataset))print(f"✓ All clients use identical preprocessing")

val_size = len(full_train_dataset) - train_sizeprint(f"\n✓ No rotation transformations applied")

print(f"  Test dataset size: {len(test_dataset)}")

train_dataset, val_dataset = torch.utils.data.random_split(print(f"  Validation dataset size: {len(val_dataset)}")

    full_train_dataset, print(f"  Train dataset size: {len(train_dataset)}")

    [train_size, val_size],print(f"\nSplit into:")

    generator=torch.Generator().manual_seed(CONFIG['seed'])
)

## Distribute Data Across Clients with Dirichlet Distribution

**Label heterogeneity via Dirichlet** - clients have non-IID label distributions controlled by alpha parameter.

- Train/Validation: Dirichlet distribution (non-IID labels)

- Test: IID distribution (standard practice for unbiased evaluation)- Higher alpha → More IID (balanced label distribution)
- Lower alpha → More non-IID (extreme label imbalance per client)

In [ ]:
# Organize data by class for each split
num_classes = 10

# Training data by class
train_indices_by_class = [[] for _ in range(num_classes)]
for idx in train_dataset.indices:
    _, label = full_train_dataset[idx]
    train_indices_by_class[label].append(idx)

# Validation data by class
val_indices_by_class = [[] for _ in range(num_classes)]
for idx in val_dataset.indices:
    _, label = full_train_dataset[idx]
    val_indices_by_class[label].append(idx)

# Test data by class
test_indices_by_class = [[] for _ in range(num_classes)]
for idx in range(len(test_dataset)):
    _, label = test_dataset[idx]
    test_indices_by_class[label].append(idx)

print(f"Training samples per class:")
for class_id in range(num_classes):
    print(f"  Class {class_id}: {len(train_indices_by_class[class_id])} samples")

print(f"\nValidation samples per class:")
for class_id in range(num_classes):
    print(f"  Class {class_id}: {len(val_indices_by_class[class_id])} samples")

print(f"\nTest samples per class:")
for class_id in range(num_classes):
    print(f"  Class {class_id}: {len(test_indices_by_class[class_id])} samples")

def distribute_with_dirichlet(indices_by_class, num_clients, alpha):
    """Distribute data to clients using Dirichlet distribution."""
    client_indices = [[] for _ in range(num_clients)]
    
    for class_id in range(len(indices_by_class)):
        class_indices = np.array(indices_by_class[class_id])
        np.random.shuffle(class_indices)
        
        # Sample proportions from Dirichlet distribution
        proportions = np.random.dirichlet(alpha=[alpha] * num_clients)
        proportions = (np.cumsum(proportions) * len(class_indices)).astype(int)[:-1]
        
        # Split indices according to proportions
        splits = np.split(class_indices, proportions)
        
        for client_idx, split in enumerate(splits):
            client_indices[client_idx].extend(split.tolist())
    
    # Shuffle each client's indices
    for client_idx in range(num_clients):
        random.shuffle(client_indices[client_idx])
    
    return client_indices

def distribute_iid(indices_by_class, num_clients):
    """Distribute data uniformly (IID) across clients."""
    all_indices = []
    for class_indices in indices_by_class:
        all_indices.extend(class_indices)
    
    random.shuffle(all_indices)
    
    # Split uniformly
    samples_per_client = len(all_indices) // num_clients
    client_indices = []
    
    for client_id in range(num_clients):
        start_idx = client_id * samples_per_client
        end_idx = start_idx + samples_per_client if client_id < num_clients - 1 else len(all_indices)
        client_indices.append(all_indices[start_idx:end_idx])
    
    return client_indices

# Apply Dirichlet distribution to training and validation data
print(f"\nApplying Dirichlet distribution with α={CONFIG['dirichlet_alpha']}...")
client_indices_train = distribute_with_dirichlet(
    train_indices_by_class, 
    CONFIG['num_clients'], 
    CONFIG['dirichlet_alpha']
)

client_indices_val = distribute_with_dirichlet(
    val_indices_by_class, 
    CONFIG['num_clients'], 
    CONFIG['dirichlet_alpha']
)

# Apply IID distribution to test data (standard practice)
print(f"Applying IID distribution to test data...")
client_indices_test = distribute_iid(
    test_indices_by_class,
    CONFIG['num_clients']
)

# Create subsets
train_subsets = [Subset(full_train_dataset, indices) for indices in client_indices_train]
val_subsets = [Subset(full_train_dataset, indices) for indices in client_indices_val]
test_subsets = [Subset(test_dataset, indices) for indices in client_indices_test]

print(f"\n{'='*60}")
print("TRAINING DATA DISTRIBUTION")
print(f"{'='*60}")
print(f"Created {len(train_subsets)} client datasets")
print(f"Average samples per client: {np.mean([len(s) for s in train_subsets]):.1f}")
print(f"Std samples per client: {np.std([len(s) for s in train_subsets]):.1f}")
print(f"Total training samples: {sum([len(s) for s in train_subsets])}")

print(f"\n{'='*60}")
print("VALIDATION DATA DISTRIBUTION")
print(f"{'='*60}")
print(f"Average samples per client: {np.mean([len(s) for s in val_subsets]):.1f}")
print(f"Std samples per client: {np.std([len(s) for s in val_subsets]):.1f}")
print(f"Total validation samples: {sum([len(s) for s in val_subsets])}")

print(f"\n{'='*60}")
print("TEST DATA DISTRIBUTION (IID)")
print(f"{'='*60}")
print(f"Average samples per client: {np.mean([len(s) for s in test_subsets]):.1f}")
print(f"Std samples per client: {np.std([len(s) for s in test_subsets]):.1f}")
print(f"Total test samples: {sum([len(s) for s in test_subsets])}")

# Calculate label distribution statistics for training data
client_label_distributions = []
for subset in train_subsets:
    label_counts = np.zeros(num_classes)
    for idx in subset.indices:
        _, label = full_train_dataset[idx]
        label_counts[label] += 1
    client_label_distributions.append(label_counts)

client_label_distributions = np.array(client_label_distributions)

# Visualize data distribution
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Plot 1: Samples per client
ax = axes[0, 0]
client_sizes = [len(subset) for subset in train_subsets]
ax.bar(range(CONFIG['num_clients']), client_sizes, alpha=0.7, edgecolor='black')
ax.axhline(y=np.mean(client_sizes), color='red', linestyle='--', 
           label=f'Mean: {np.mean(client_sizes):.0f}')
ax.set_xlabel('Client ID', fontsize=12)
ax.set_ylabel('Number of Samples', fontsize=12)
ax.set_title('Training Samples per Client (Dirichlet Distribution)', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

# Plot 2: Label distribution heatmap
ax = axes[0, 1]
im = ax.imshow(client_label_distributions.T, aspect='auto', cmap='YlOrRd', interpolation='nearest')
ax.set_xlabel('Client ID', fontsize=12)
ax.set_ylabel('Class Label', fontsize=12)
ax.set_title(f'Training Label Distribution Heatmap (α={CONFIG["dirichlet_alpha"]})', fontsize=14, fontweight='bold')
ax.set_xticks(range(0, CONFIG['num_clients'], max(1, CONFIG['num_clients']//10)))
ax.set_yticks(range(num_classes))
plt.colorbar(im, ax=ax, label='Sample Count')

# Plot 3: Classes per client
ax = axes[1, 0]
classes_per_client = [(dist > 0).sum() for dist in client_label_distributions]
ax.hist(classes_per_client, bins=range(1, num_classes + 2), alpha=0.7, edgecolor='black', color='steelblue')
ax.set_xlabel('Number of Classes', fontsize=12)
ax.set_ylabel('Number of Clients', fontsize=12)
ax.set_title('Classes per Client Distribution', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
ax.text(0.02, 0.98, f'Mean: {np.mean(classes_per_client):.1f}\nStd: {np.std(classes_per_client):.1f}',
        transform=ax.transAxes, fontsize=11, verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# Plot 4: Label entropy per client
ax = axes[1, 1]
entropies = []
for dist in client_label_distributions:
    probs = dist / dist.sum()
    probs = probs[probs > 0]  # Remove zeros
    entropy = -np.sum(probs * np.log2(probs))
    entropies.append(entropy)

ax.hist(entropies, bins=20, alpha=0.7, edgecolor='black', color='coral')
ax.axvline(x=np.mean(entropies), color='red', linestyle='--', linewidth=2,
           label=f'Mean: {np.mean(entropies):.2f}')
ax.axvline(x=np.log2(num_classes), color='green', linestyle='--', linewidth=2,
           label=f'Max (uniform): {np.log2(num_classes):.2f}')
ax.set_xlabel('Label Entropy (bits)', fontsize=12)
ax.set_ylabel('Number of Clients', fontsize=12)
ax.set_title('Training Label Entropy Distribution', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('fedavg_no_rotation_dirichlet_data_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\n✓ Train/Val: Dirichlet distribution (α={CONFIG['dirichlet_alpha']}) - label heterogeneity")
print(f"✓ Test: IID distribution - unbiased evaluation")
print(f"✓ Mean label entropy: {np.mean(entropies):.2f} bits (max: {np.log2(num_classes):.2f})")
print(f"✓ Mean classes per client: {np.mean(classes_per_client):.1f} / {num_classes}")

## Initialize Global Model

In [ ]:
# Initialize global model
global_model = get_model(
    model_name=CONFIG['model_name'],
    num_classes=10,
    pretrained=CONFIG['pretrained']
).to(device)

criterion = nn.CrossEntropyLoss()

print(f"Global model initialized: {CONFIG['model_name']}")
print(f"Number of parameters: {sum(p.numel() for p in global_model.parameters()):,}")

print(f"Test dataset size: {len(test_dataset)}")

# Create validation loader (for training curves)print(f"\nValidation dataset size: {sum([len(s) for s in val_subsets])}")

val_loader = DataLoader(

    torch.utils.data.ConcatDataset(val_subsets),)

    batch_size=CONFIG['batch_size'],    shuffle=False

    shuffle=False    batch_size=CONFIG['batch_size'],

)    test_dataset,

test_loader = DataLoader(
# Create test loader (for final evaluation only)

## Define FedAvg Functions

In [ ]:
def train_local_model(model, train_loader, epochs, lr):
    """Train local model for specified epochs."""
    model.train()
    optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    
    for epoch in range(epochs):
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
    
    return model.state_dict()

def aggregate_models(client_weights, client_sizes):
    """Aggregate client models using weighted averaging."""
    total_size = sum(client_sizes)
    avg_weights = copy.deepcopy(client_weights[0])
    
    for key in avg_weights.keys():
        avg_weights[key] = torch.zeros_like(avg_weights[key], dtype=torch.float32)
        for i in range(len(client_weights)):
            weight = client_sizes[i] / total_size
            avg_weights[key] += client_weights[i][key] * weight
    
    return avg_weights

def evaluate_model(model, data_loader):
    """Evaluate model on given data loader."""
    model.eval()
    test_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for inputs, labels in data_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            test_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    
    avg_loss = test_loss / len(data_loader)
    accuracy = correct / total
    
    return avg_loss, accuracy

print("FedAvg functions defined")

## Run FedAvg Training

In [ ]:
# FedAvg configuration
num_rounds = CONFIG['training_rounds']
local_epochs = CONFIG['warmup_epochs']
client_fraction = CONFIG['client_fraction']
num_selected = max(int(client_fraction * CONFIG['num_clients']), 1)

# Storage for results
val_losses = []
val_accs = []
round_times = []

print(f"{'='*70}")
print(f"FEDAVG TRAINING (NO ROTATIONS)")
print(f"{'='*70}")
print(f"Number of rounds: {num_rounds}")
print(f"Local epochs: {local_epochs}")
print(f"Clients per round: {num_selected}/{CONFIG['num_clients']} ({client_fraction*100:.0f}%)")
print(f"Learning rate: {CONFIG['lr']}")
print(f"Batch size: {CONFIG['batch_size']}")
print(f"{'='*70}")
print(f"NOTE: Using VALIDATION set for training curves")
print(f"      Test set will be used only for final evaluation")
print(f"{'='*70}\n")

total_start_time = time.time()

for round_num in range(1, num_rounds + 1):
    round_start = time.time()
    
    # Sample clients for this round
    selected_clients = random.sample(range(CONFIG['num_clients']), num_selected)
    
    # Store client updates
    client_weights = []
    client_sizes = []
    
    # Get global weights
    global_weights = global_model.state_dict()
    
    # Train selected clients
    for client_idx in selected_clients:
        # Create local model
        local_model = get_model(
            model_name=CONFIG['model_name'],
            num_classes=10,
            pretrained=False
        ).to(device)
        local_model.load_state_dict(global_weights)
        
        # Create data loader
        train_loader = DataLoader(
            train_subsets[client_idx],
            batch_size=CONFIG['batch_size'],
            shuffle=True
        )
        
        # Train locally
        updated_weights = train_local_model(local_model, train_loader, local_epochs, CONFIG['lr'])
        
        # Store update
        client_weights.append(updated_weights)
        client_sizes.append(len(train_subsets[client_idx]))
    
    # Aggregate updates
    # Evaluate on VALIDATION set
    val_loss, val_acc = evaluate_model(global_model, val_loader)
    val_losses.append(val_loss)
    val_accs.append(val_acc)
    test_loss, test_acc = evaluate_model(global_model, test_loader)
    test_losses.append(test_loss)
    test_accs.append(test_acc)
    
    round_time = time.time() - round_start
    round_times.append(round_time)
    
              f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}, "
    if round_num % 5 == 0 or round_num == 1:
        print(f"Round {round_num}/{num_rounds} - "
              f"Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.4f}, "
              f"Time: {round_time:.2f}s")

total_training_time = time.time() - total_start_time

print(f"\n{'='*70}")
print(f"FedAvg Training Complete!")
print(f"Best validation accuracy: {max(val_accs):.4f} (round {np.argmax(val_accs)+1})")


# Final evaluation on TEST set
print(f"Average time per round: {np.mean(round_times):.2f}s")print(f"{'='*70}")

print(f"\n{'='*70}")
print(f"Final test accuracy: {test_accs[-1]:.4f}")print(f"Test Accuracy: {test_acc:.4f}")

print(f"FINAL EVALUATION ON TEST SET")
print(f"Best test accuracy: {max(test_accs):.4f} (round {np.argmax(test_accs)+1})")print(f"Test Loss: {test_loss:.4f}")

print(f"{'='*70}")test_loss, test_acc = evaluate_model(global_model, test_loader)

## Visualize Training Progress

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Plot 1: Loss curve
ax = axes[0]
rounds_range = range(1, num_rounds + 1)
ax.plot(rounds_range, val_losses, 'o-', linewidth=2, markersize=4)
ax.set_xlabel('Round', fontsize=12)
ax.set_ylabel('Validation Loss', fontsize=12)
ax.set_title('FedAvg Validation Loss (No Rotations)', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)

# Plot 2: Accuracy curve
ax = axes[1]
ax.plot(rounds_range, val_accs, 's-', linewidth=2, markersize=4, color='green', label='Validation')
ax.axhline(y=max(val_accs), color='orange', linestyle='--', alpha=0.5, 
           label=f'Best Val: {max(val_accs):.4f}')
ax.axhline(y=test_acc, color='red', linestyle='--', alpha=0.7, linewidth=2,
           label=f'Final Test: {test_acc:.4f}')
ax.set_xlabel('Round', fontsize=12)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('FedAvg Validation Accuracy (No Rotations)', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim([0, 1])

plt.tight_layout()
plt.savefig('fedavg_no_rotation_training_curves.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"      Final test accuracy: {test_acc:.4f}")

print("Training curves saved")print(f"Note: Curves show validation performance during training")

## Confusion Matrix Analysis

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import itertools

# CIFAR-10 class names
class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer', 
               'dog', 'frog', 'horse', 'ship', 'truck']

# Collect all predictions and true labels
all_predictions = []
all_labels = []

global_model.eval()
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = global_model(inputs)
        _, predicted = outputs.max(1)
        
        all_predictions.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Compute confusion matrix
cm = confusion_matrix(all_labels, all_predictions)
cm_percent = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100

# Plot confusion matrix
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Plot 1: Counts
ax = axes[0]
im1 = ax.imshow(cm, interpolation='nearest', cmap='Blues')
ax.figure.colorbar(im1, ax=ax)
ax.set(xticks=np.arange(cm.shape[1]),
       yticks=np.arange(cm.shape[0]),
       xticklabels=class_names,
       yticklabels=class_names,
       xlabel='Predicted Label',
       ylabel='True Label',
       title='Confusion Matrix - Counts\nFedAvg (No Rotations)')
plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")

thresh = cm.max() / 2.
for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
    ax.text(j, i, format(cm[i, j], 'd'),
            ha="center", va="center",
            color="white" if cm[i, j] > thresh else "black",
            fontsize=9)

# Plot 2: Percentages
ax = axes[1]
im2 = ax.imshow(cm_percent, interpolation='nearest', cmap='Blues')
ax.figure.colorbar(im2, ax=ax, format='%.1f%%')
ax.set(xticks=np.arange(cm.shape[1]),
       yticks=np.arange(cm.shape[0]),
       xticklabels=class_names,
       yticklabels=class_names,
       xlabel='Predicted Label',
       ylabel='True Label',
       title='Confusion Matrix - Percentages\nFedAvg (No Rotations)')
plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")

thresh = cm_percent.max() / 2.
for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
    ax.text(j, i, format(cm_percent[i, j], '.1f'),
            ha="center", va="center",
            color="white" if cm_percent[i, j] > thresh else "black",
            fontsize=9)

plt.tight_layout()
plt.savefig('fedavg_no_rotation_confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

print("Confusion matrix saved")

## Per-Class Performance Analysis

In [ ]:
# Detailed classification report
print(f"\n{'='*70}")
print("CLASSIFICATION REPORT")
print(f"{'='*70}\n")
print(classification_report(all_labels, all_predictions, target_names=class_names, digits=4))

# Per-class accuracy visualization
per_class_accuracy = cm.diagonal() / cm.sum(axis=1)

plt.figure(figsize=(12, 6))
bars = plt.bar(class_names, per_class_accuracy, alpha=0.7, edgecolor='black', color='steelblue')
plt.axhline(y=np.mean(per_class_accuracy), color='red', linestyle='--', 
            label=f'Mean: {np.mean(per_class_accuracy):.4f}')

# Color bars based on performance
colors = ['green' if acc > 0.7 else 'orange' if acc > 0.5 else 'red' 
          for acc in per_class_accuracy]
for bar, color in zip(bars, colors):
    bar.set_color(color)
    bar.set_alpha(0.7)

plt.xlabel('Class', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.title('Per-Class Accuracy - FedAvg (No Rotations)', fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.legend()
plt.grid(True, alpha=0.3, axis='y')
plt.ylim([0, 1])
plt.tight_layout()
plt.savefig('fedavg_no_rotation_per_class_accuracy.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\nPer-class accuracy statistics:")
print(f"  Mean: {np.mean(per_class_accuracy):.4f}")
print(f"  Std:  {np.std(per_class_accuracy):.4f}")
print(f"  Min:  {np.min(per_class_accuracy):.4f} ({class_names[np.argmin(per_class_accuracy)]})")
print(f"  Max:  {np.max(per_class_accuracy):.4f} ({class_names[np.argmax(per_class_accuracy)]})")

## Most Confused Class Pairs

In [ ]:
# Find most confused pairs
print(f"\n{'='*70}")
print("MOST CONFUSED CLASS PAIRS")
print(f"{'='*70}\n")

confused_pairs = []
for i in range(len(class_names)):
    for j in range(len(class_names)):
        if i != j and cm[i, j] > 0:
            confused_pairs.append((class_names[i], class_names[j], cm[i, j]))

confused_pairs.sort(key=lambda x: x[2], reverse=True)

print(f"{'True Class':<15} {'Predicted As':<15} {'Count':<10} {'% of True Class':<15}")
print("-"*60)
for true_class, pred_class, count in confused_pairs[:15]:
    true_idx = class_names.index(true_class)
    total_true = cm[true_idx, :].sum()
    percentage = (count / total_true) * 100
    print(f"{true_class:<15} {pred_class:<15} {count:<10} {percentage:>6.2f}%")

## Save Results

In [ ]:
# Save results to file
results = {
    'method': 'fedavg_no_rotation_dirichlet',
    'config': CONFIG,
    'data_distribution': 'Dirichlet',
    'dirichlet_alpha': CONFIG['dirichlet_alpha'],
    'rotations_applied': False,
    'num_rounds': num_rounds,
    'local_epochs': local_epochs,
    'client_fraction': client_fraction,
    'training_time': total_training_time,
    'avg_round_time': float(np.mean(round_times)),
    'best_val_acc': float(max(val_accs)),
    'best_val_round': int(np.argmax(val_accs) + 1),
    'final_test_acc': float(test_acc),
    'final_test_loss': float(test_loss),
    'val_losses': [float(x) for x in val_losses],
    'val_accs': [float(x) for x in val_accs],
    'per_class_accuracy': [float(x) for x in per_class_accuracy],
    'confusion_matrix': cm.tolist(),
    'data_stats': {
        'mean_samples_per_client': float(np.mean([len(s) for s in train_subsets])),
        'std_samples_per_client': float(np.std([len(s) for s in train_subsets])),
        'mean_label_entropy': float(np.mean(entropies)),
        'std_label_entropy': float(np.std(entropies)),
        'mean_classes_per_client': float(np.mean(classes_per_client))
    }
}

with open(f'fedavg_no_rotation_alpha{CONFIG["dirichlet_alpha"]}_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print(f"Results saved to 'fedavg_no_rotation_alpha{CONFIG['dirichlet_alpha']}_results.json'")

# Save model checkpoint
torch.save({
    'round': num_rounds,
    'model_state_dict': global_model.state_dict(),
    'test_acc': test_acc,
    'best_val_acc': max(val_accs),
    'config': CONFIG
}, f'fedavg_no_rotation_alpha{CONFIG["dirichlet_alpha"]}_checkpoint.pth')

print(f"Model checkpoint saved to 'fedavg_no_rotation_alpha{CONFIG['dirichlet_alpha']}_checkpoint.pth'")
print("\n✓ All results and visualizations saved")

## Summary

**Federated Averaging (FedAvg) - No Rotations, Dirichlet Label Distribution:**

**Key Characteristics:**
- ✓ No rotation transformations applied (no feature heterogeneity)
- ✓ All clients use identical standard preprocessing  
- ✓ **Dirichlet label distribution** (non-IID, controlled by α parameter)
- ✓ Isolates label heterogeneity from feature heterogeneity

**Heterogeneity:**
- **Label heterogeneity**: Dirichlet α={CONFIG['dirichlet_alpha']}
  - Lower α → More extreme label imbalance per client
  - Higher α → More balanced label distribution
- **No feature heterogeneity**: All clients see same image transformations

**Expected Results:**
- **Performance depends on α**: Lower α → harder training, lower accuracy
- **Conflicting gradients**: Clients with different label distributions produce divergent updates
- **Slower convergence**: Compared to IID, non-IID label distributions slow down learning
- **Better than rotation + Dirichlet**: Only one type of heterogeneity (easier than dual)

**Comparison Goals:**

- vs **FedAvg with Rotations + Dirichlet**: Impact of adding feature heterogeneityRun this notebook with different α values (0.1, 0.5, 1.0, 10.0) to measure FedAvg's robustness to label imbalance.

- vs **FedAvg IID**: Cost of label heterogeneity alone**Alpha Sensitivity:**

- vs **Ensemble (Dirichlet only)**: Flat vs hierarchical on label-heterogeneous data

- vs **Centralized**: Total cost of federated learning with label heterogeneityIsolates the impact of label heterogeneity on FedAvg performance without confounding effects from rotation-based feature heterogeneity. Shows how non-IID label distributions alone affect federated learning convergence and accuracy.

**Research Value:**